# Transcrição do áudio — Seminário I

Transcrição com `faster-whisper` (CPU), em português.

## 1. Configuração

In [ ]:
from pathlib import Path

PASTA = Path.cwd()  # 07_revisoes_banca/seminario_I
AUDIO = PASTA / "audio" / "NOME_DO_ARQUIVO.m4a"  # <- ajustar
SAIDA = PASTA / "transcricao"
SAIDA.mkdir(exist_ok=True)

MODELO = "medium"   # tiny | base | small | medium | large-v3
IDIOMA = "pt"

print(AUDIO, AUDIO.exists())

## 2. Carregar o modelo

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel(MODELO, device="cpu", compute_type="int8")

## 3. Transcrever

In [ ]:
segments, info = model.transcribe(
    str(AUDIO),
    language=IDIOMA,
    vad_filter=True,
    beam_size=5,
)
print(f"Duração: {info.duration/60:.1f} min")

segmentos = []
for s in segments:
    segmentos.append(s)
    print(f"[{s.start:7.1f}s] {s.text.strip()}")

## 4. Salvar

In [ ]:
def fmt(t):
    h, r = divmod(int(t), 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

base = SAIDA / AUDIO.stem
(base.with_suffix(".txt")).write_text(
    "\n".join(s.text.strip() for s in segmentos), encoding="utf-8")
(base.with_name(base.name + "_tempos.txt")).write_text(
    "\n".join(f"[{fmt(s.start)} - {fmt(s.end)}] {s.text.strip()}" for s in segmentos),
    encoding="utf-8")
print("Salvo em", SAIDA)